In [ ]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "D:/pyspark_udemy_codespace/setup/spark-warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

Spark version: 3.5.0


In [ ]:
from pyspark.sql.functions import concat_ws # type: ignore

data_list = [(2022, 5, 18) , (2019, 12, 31), (2021, 1, 1), (2020, 1, 1), 
             (2022, 1, 1), (2023, 5, 25),   (2020, 5, 25),   (2021, 2, 29)]

df = spark.createDataFrame(data_list).toDF('Y', 'M', 'D')\
        .withColumn("date_str", concat_ws('-', 'Y', 'M', 'D'))
df.show()

+----+---+---+----------+
|   Y|  M|  D|  date_str|
+----+---+---+----------+
|2022|  5| 18| 2022-5-18|
|2019| 12| 31|2019-12-31|
|2021|  1|  1|  2021-1-1|
|2020|  1|  1|  2020-1-1|
|2022|  1|  1|  2022-1-1|
|2023|  5| 25| 2023-5-25|
|2020|  5| 25| 2020-5-25|
|2021|  2| 29| 2021-2-29|
+----+---+---+----------+



In [ ]:
"""
2. Convert strings to date
    Spark validates the date against the Proleptic Gregorian calendar.
    The negative years are BC, and the positive values are AD in Gregorian calander.
    Valid dates are taken, and invalid dates throw an exception or taken as null
"""
from pyspark.sql.functions import expr, col, to_date # type: ignore

df = df.withColumn("valid_date", to_date(col('date_str'), 'y-M-d'))
# df = df.withColumn("valid_date", expr)
df.show()
df.printSchema()

+----+---+---+----------+----------+
|   Y|  M|  D|  date_str|valid_date|
+----+---+---+----------+----------+
|2022|  5| 18| 2022-5-18|2022-05-18|
|2019| 12| 31|2019-12-31|2019-12-31|
|2021|  1|  1|  2021-1-1|2021-01-01|
|2020|  1|  1|  2020-1-1|2020-01-01|
|2022|  1|  1|  2022-1-1|2022-01-01|
|2023|  5| 25| 2023-5-25|2023-05-25|
|2020|  5| 25| 2020-5-25|2020-05-25|
|2021|  2| 29| 2021-2-29|      NULL|
+----+---+---+----------+----------+

root
 |-- Y: long (nullable = true)
 |-- M: long (nullable = true)
 |-- D: long (nullable = true)
 |-- date_str: string (nullable = false)
 |-- valid_date: date (nullable = true)



In [ ]:
"""
Add Subtract days and months to date
"""
from pyspark.sql.functions import date_add, date_sub, add_months # type: ignore

df = df.withColumns({
    "add_5_days":   date_add(col("valid_date"), 5),
    "sub_5_days":   date_sub(col("valid_date"), 5),
    "add_5_months": add_months(col("valid_date"), 5),
    "sub_5_months": add_months(col("valid_date"), -5)
})

df.show()
df.printSchema()

+----+---+---+----------+----------+----------+----------+------------+------------+
|   Y|  M|  D|  date_str|valid_date|add_5_days|sub_5_days|add_5_months|sub_5_months|
+----+---+---+----------+----------+----------+----------+------------+------------+
|2022|  5| 18| 2022-5-18|2022-05-18|2022-05-23|2022-05-13|  2022-10-18|  2021-12-18|
|2019| 12| 31|2019-12-31|2019-12-31|2020-01-05|2019-12-26|  2020-05-31|  2019-07-31|
|2021|  1|  1|  2021-1-1|2021-01-01|2021-01-06|2020-12-27|  2021-06-01|  2020-08-01|
|2020|  1|  1|  2020-1-1|2020-01-01|2020-01-06|2019-12-27|  2020-06-01|  2019-08-01|
|2022|  1|  1|  2022-1-1|2022-01-01|2022-01-06|2021-12-27|  2022-06-01|  2021-08-01|
|2023|  5| 25| 2023-5-25|2023-05-25|2023-05-30|2023-05-20|  2023-10-25|  2022-12-25|
|2020|  5| 25| 2020-5-25|2020-05-25|2020-05-30|2020-05-20|  2020-10-25|  2019-12-25|
|2021|  2| 29| 2021-2-29|      NULL|      NULL|      NULL|        NULL|        NULL|
+----+---+---+----------+----------+----------+----------+-------

In [10]:
"""
Current date, date difference, interval
"""
from pyspark.sql.functions import current_date, date_diff # type: ignore

df = df.withColumns({
    "current_date": current_date(),
    "delta_date_days": date_diff(col("add_5_months"), col("valid_date")),
    "delta_date_interval": col("current_date") - col("valid_date")
})
df.show()

+----+---+---+----------+----------+----------+----------+------------+------------+------------+---------------+-------------------+
|   Y|  M|  D|  date_str|valid_date|add_5_days|sub_5_days|add_5_months|sub_5_months|current_date|delta_date_days|delta_date_interval|
+----+---+---+----------+----------+----------+----------+------------+------------+------------+---------------+-------------------+
|2022|  5| 18| 2022-5-18|2022-05-18|2022-05-23|2022-05-13|  2022-10-18|  2021-12-18|  2026-05-20|            153|INTERVAL '1463' DAY|
|2019| 12| 31|2019-12-31|2019-12-31|2020-01-05|2019-12-26|  2020-05-31|  2019-07-31|  2026-05-20|            152|INTERVAL '2332' DAY|
|2021|  1|  1|  2021-1-1|2021-01-01|2021-01-06|2020-12-27|  2021-06-01|  2020-08-01|  2026-05-20|            151|INTERVAL '1965' DAY|
|2020|  1|  1|  2020-1-1|2020-01-01|2020-01-06|2019-12-27|  2020-06-01|  2019-08-01|  2026-05-20|            152|INTERVAL '2331' DAY|
|2022|  1|  1|  2022-1-1|2022-01-01|2022-01-06|2021-12-27|  20

In [13]:
"""
Format Date
"""

from pyspark.sql.functions import date_format # type: ignore

df = df.withColumn("fmt_date", date_format(col("valid_date"), "dd MMM yyyy")) # datatype will be string
df.show()
df.printSchema()

+----+---+---+----------+----------+----------+----------+------------+------------+------------+---------------+-------------------+-----------+
|   Y|  M|  D|  date_str|valid_date|add_5_days|sub_5_days|add_5_months|sub_5_months|current_date|delta_date_days|delta_date_interval|   fmt_date|
+----+---+---+----------+----------+----------+----------+------------+------------+------------+---------------+-------------------+-----------+
|2022|  5| 18| 2022-5-18|2022-05-18|2022-05-23|2022-05-13|  2022-10-18|  2021-12-18|  2026-05-20|            153|INTERVAL '1463' DAY|18 May 2022|
|2019| 12| 31|2019-12-31|2019-12-31|2020-01-05|2019-12-26|  2020-05-31|  2019-07-31|  2026-05-20|            152|INTERVAL '2332' DAY|31 Dec 2019|
|2021|  1|  1|  2021-1-1|2021-01-01|2021-01-06|2020-12-27|  2021-06-01|  2020-08-01|  2026-05-20|            151|INTERVAL '1965' DAY|01 Jan 2021|
|2020|  1|  1|  2020-1-1|2020-01-01|2020-01-06|2019-12-27|  2020-06-01|  2019-08-01|  2026-05-20|            152|INTERVAL '2